# AI for Radiologic Image Tasks: Detection

This notebook presents an educational workflow for object detection in chest radiographs. It uses the Montgomery dataset and converts the available lung masks into bounding-box annotations suitable for model development.

The objective is to localize the left and right lung fields. In contrast to image-level classification, the output includes the spatial extent of each target structure.

> **Google Colab quick start:** open this notebook in Colab, select **Runtime → Change runtime type → T4 GPU** (recommended for training), then choose **Runtime → Run all**. The setup cell installs missing libraries and downloads the public dataset automatically; no manual uploads or path edits are needed.



In [ ]:
import random
import shutil
import warnings

import subprocess
import sys
import urllib.request
import zipfile
from pathlib import Path

import cv2
import matplotlib.patches as patches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import yaml
from ultralytics import YOLO
from tqdm.auto import tqdm

## 0. One-click setup

Run the next cell exactly as it is. It automatically installs any missing libraries, downloads the public dataset(s) into the current runtime, and reuses them if you run the cell again. In Colab, choose a GPU before running all cells for a much faster training run.

The downloaded data and model results live only for this Colab session unless you explicitly save them elsewhere.


In [ ]:
IN_COLAB = "google.colab" in sys.modules

def install(packages):
    """Install the small set of extra libraries used by this notebook."""
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *packages])

DATASET_URL = "https://huggingface.co/datasets/Famatsu123/montgomery-shenzhen-tuberculosis-cxr/resolve/main"
DATA_ROOT = (Path("/content") if IN_COLAB else Path.cwd()) / "datasets"

def get_dataset(archive_name, folder):
    """Download one public archive and extract it once; later runs reuse it."""
    if folder.exists():
        return folder
    DATA_ROOT.mkdir(parents=True, exist_ok=True)
    archive = DATA_ROOT / archive_name
    if not archive.exists():
        print(f"Downloading {archive_name} from Hugging Face (first run only)...")
        urllib.request.urlretrieve(f"{DATASET_URL}/{archive_name}?download=true", archive)
    print(f"Extracting {archive_name}...")
    with zipfile.ZipFile(archive) as zip_file:
        zip_file.extractall(DATA_ROOT)
    return folder

PACKAGES = ["ultralytics", "opencv-python-headless", "pyyaml"]
install(PACKAGES)

DATASET_DIR = get_dataset("MontgomerySet.zip", DATA_ROOT / "MontgomerySet")
IMAGE_DIR = DATASET_DIR / "CXR_png"
LEFT_MASK_DIR = DATASET_DIR / "ManualMask" / "leftMask"
RIGHT_MASK_DIR = DATASET_DIR / "ManualMask" / "rightMask"

warnings.filterwarnings("ignore", category=UserWarning)
plt.rcParams["figure.figsize"] = (8, 8)
plt.rcParams["image.cmap"] = "gray"

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

assert IMAGE_DIR.exists(), f"Image directory not found: {IMAGE_DIR}"

YOLO_DATASET_DIR = RESULTS_DIR / "montgomery_lung_yolo"
DATA_YAML_PATH = YOLO_DATASET_DIR / "montgomery_lung.yaml"
assert LEFT_MASK_DIR.exists(), f"Left masks not found: {LEFT_MASK_DIR}"
assert RIGHT_MASK_DIR.exists(), f"Right masks not found: {RIGHT_MASK_DIR}"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using compute device: {DEVICE}")
print("Dataset is ready. Re-running this cell reuses the downloaded files.")


## 1. Build Image And Mask Index

The Montgomery dataset stores lung masks separately from the X-ray images. This section builds a table where each row links one radiograph to its left-lung mask, right-lung mask, and disease label.

The disease label is retained only as metadata for stratified splitting. It is not the detection target; the model is trained to localize lung fields rather than classify normal and abnormal studies.

In [ ]:
def disease_label_from_id(image_id: str) -> int:
    return int(image_id.rsplit("_", 1)[-1])


def build_montgomery_index() -> pd.DataFrame:
    rows = []
    for image_path in sorted(IMAGE_DIR.glob("*.png")):
        left_mask_path = LEFT_MASK_DIR / image_path.name
        right_mask_path = RIGHT_MASK_DIR / image_path.name
        if not left_mask_path.exists() or not right_mask_path.exists():
            continue
        image_id = image_path.stem
        rows.append({
            "ImageId": image_id,
            "image_path": image_path,
            "left_mask_path": left_mask_path,
            "right_mask_path": right_mask_path,
            "disease_label": disease_label_from_id(image_id),
        })
    return pd.DataFrame(rows)


images_df = build_montgomery_index()
assert not images_df.empty, "No complete image/left-mask/right-mask triplets found."
print(f"Complete image/mask pairs found for YOLO detection: {len(images_df):,}")
print("Disease-label counts kept only as descriptive metadata:")
images_df["disease_label"].value_counts().rename({0: "normal", 1: "abnormal"})


## 2. Convert Masks To Boxes

The original Montgomery annotations are segmentation masks. For object detection, each left- and right-lung mask is converted into the smallest rectangle containing the annotated pixels.

Each image therefore has two target boxes, one for each lung. Both boxes represent the same anatomical class because the objective is localization rather than side classification.

In [ ]:
def read_grayscale_png(path: Path) -> np.ndarray:
    image = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if image is None:
        raise FileNotFoundError(path)
    return image


def normalize_percentile(image: np.ndarray, lower: float = 1, upper: float = 99, eps: float = 1e-7) -> np.ndarray:
    image = image.astype(np.float32)
    lo, hi = np.percentile(image, [lower, upper])
    image = np.clip(image, lo, hi)
    return ((image - lo) / (hi - lo + eps)).astype(np.float32)


def mask_to_box(mask: np.ndarray) -> list[float] | None:
    ys, xs = np.where(mask > 0)
    if len(xs) == 0 or len(ys) == 0:
        return None
    x_min = float(xs.min())
    y_min = float(ys.min())
    x_max = float(xs.max() + 1)
    y_max = float(ys.max() + 1)
    return [x_min, y_min, x_max, y_max]


def lung_boxes(row: pd.Series) -> np.ndarray:
    boxes = []
    for mask_path in [row.left_mask_path, row.right_mask_path]:
        box = mask_to_box(read_grayscale_png(mask_path))
        if box is not None:
            boxes.append(box)
    return np.array(boxes, dtype=np.float32)


def image_and_boxes(image_id: str) -> tuple[np.ndarray, np.ndarray]:
    row = images_df.loc[images_df["ImageId"] == image_id].iloc[0]
    image = read_grayscale_png(row.image_path)
    boxes = lung_boxes(row)
    return image, boxes


In [ ]:
box_rows = []
for row in images_df.itertuples(index=False):
    boxes = lung_boxes(pd.Series(row._asdict()))
    box_rows.append({
        "ImageId": row.ImageId,
        "boxes": boxes,
        "n_boxes": len(boxes),
    })

boxes_df = images_df.merge(pd.DataFrame(box_rows), on="ImageId")
assert (boxes_df["n_boxes"] == 2).all(), "Each Montgomery image should have two lung boxes."
print("First mask-derived box records; n_boxes should be 2 for paired lung boxes:")
boxes_df[["ImageId", "n_boxes", "disease_label"]].head()


In [ ]:
def draw_boxes(ax, boxes, color="lime", linewidth=2, labels=None, scores=None):
    for i, box in enumerate(boxes):
        x_min, y_min, x_max, y_max = box
        rect = patches.Rectangle(
            (x_min, y_min),
            x_max - x_min,
            y_max - y_min,
            linewidth=linewidth,
            edgecolor=color,
            facecolor="none",
        )
        ax.add_patch(rect)
        if labels is not None or scores is not None:
            label = labels[i] if labels is not None else "box"
            score = f" {scores[i]:.2f}" if scores is not None else ""
            ax.text(x_min, y_min, f"{label}{score}", color=color, fontsize=9, va="bottom")


sample_row = boxes_df.sample(1, random_state=SEED).iloc[0]
sample_image, sample_boxes = image_and_boxes(sample_row.ImageId)

print(f"Random sample selected for box-overlay sanity check: {sample_row.ImageId}")
print(f"Original grayscale image shape before YOLO resizing: {sample_image.shape}")
print(f"Mask-derived lung boxes in original pixel coordinates:\\n{sample_boxes}")

fig, ax = plt.subplots(1, 1, figsize=(7, 7))
ax.imshow(normalize_percentile(sample_image), cmap="gray")
draw_boxes(ax, sample_boxes)
ax.set_title("Mask-derived lung boxes")
ax.axis("off")
plt.tight_layout()


In [ ]:
step_row = boxes_df.sample(1, random_state=SEED).iloc[0]
step_metadata = images_df.loc[images_df["ImageId"] == step_row.ImageId].iloc[0]

step_image = read_grayscale_png(step_metadata.image_path)
left_mask = read_grayscale_png(step_metadata.left_mask_path) > 0
right_mask = read_grayscale_png(step_metadata.right_mask_path) > 0
combined_mask = left_mask | right_mask
step_boxes = lung_boxes(step_metadata)

display_image = normalize_percentile(step_image)
mask_overlay = np.ma.masked_where(~combined_mask, combined_mask)

fig, axes = plt.subplots(1, 4, figsize=(18, 5))

axes[0].imshow(display_image, cmap="gray")
axes[0].set_title("1. Chest X-ray")

axes[1].imshow(left_mask, cmap="Blues")
axes[1].imshow(right_mask, cmap="Greens", alpha=0.7)
axes[1].set_title("2. Left and right masks")

axes[2].imshow(display_image, cmap="gray")
axes[2].imshow(mask_overlay, cmap="autumn", alpha=0.35)
axes[2].set_title("3. Masks over image")

axes[3].imshow(display_image, cmap="gray")
draw_boxes(axes[3], step_boxes, color="cyan", linewidth=2, labels=["lung", "lung"])
axes[3].set_title("4. Final bounding boxes")

for ax in axes:
    ax.axis("off")

plt.suptitle(f"Bounding-box generation: {step_row.ImageId}", y=1.02)
plt.tight_layout()

pd.DataFrame(step_boxes, columns=["x_min", "y_min", "x_max", "y_max"]).assign(class_name="lung")


## 3. Train/Validation/Test Split

The dataset is divided into training, validation, and test subsets. The training set is used to update the model weights, the validation set is used during training to monitor performance, and the test set is kept aside for the final evaluation.

We keep the same stratified split idea as the original detection notebook. The disease labels are not target classes here; they only keep the split balanced across normal and abnormal studies.

In [ ]:
def stratified_image_split(
    df: pd.DataFrame,
    valid_fraction: float = 0.15,
    test_fraction: float = 0.15,
    seed: int = SEED,
):
    if valid_fraction <= 0 or test_fraction <= 0 or valid_fraction + test_fraction >= 1:
        raise ValueError("valid_fraction and test_fraction must be positive and sum to less than 1.")

    rng = np.random.default_rng(seed)
    train_ids, valid_ids, test_ids = [], [], []
    for _, group in df.groupby("disease_label"):
        ids = group["ImageId"].to_numpy().copy()
        rng.shuffle(ids)
        n_valid = max(1, int(round(len(ids) * valid_fraction)))
        n_test = max(1, int(round(len(ids) * test_fraction)))
        if n_valid + n_test >= len(ids):
            raise ValueError("Not enough images in each class to create train, validation, and test splits.")
        valid_ids.extend(ids[:n_valid])
        test_ids.extend(ids[n_valid:n_valid + n_test])
        train_ids.extend(ids[n_valid + n_test:])

    train_ids = np.array(train_ids)
    valid_ids = np.array(valid_ids)
    test_ids = np.array(test_ids)
    rng.shuffle(train_ids)
    rng.shuffle(valid_ids)
    rng.shuffle(test_ids)
    return train_ids, valid_ids, test_ids


train_ids, valid_ids, test_ids = stratified_image_split(boxes_df)
label_lookup = boxes_df.set_index("ImageId")["disease_label"]
print("Stratified train/validation/test split size summary:")
pd.DataFrame({
    "split": ["all", "train", "valid", "test"],
    "images": [len(boxes_df), len(train_ids), len(valid_ids), len(test_ids)],
    "abnormal_fraction": [
        boxes_df["disease_label"].mean(),
        label_lookup.loc[train_ids].mean(),
        label_lookup.loc[valid_ids].mean(),
        label_lookup.loc[test_ids].mean(),
    ],
})


## 4. Prepare the Detection Dataset

The detection framework expects a predefined folder structure with separate image and annotation directories for each subset. This section prepares the images and writes one annotation file per image.

Each annotation records a class identifier and the normalized center coordinates, width, and height of a bounding box. Normalized coordinates preserve the relative spatial location of the target across images of different dimensions.

In [ ]:
def xyxy_to_yolo_line(box: np.ndarray, image_width: int, image_height: int, class_id: int = 0) -> str:
    x_min, y_min, x_max, y_max = box.astype(float)
    x_center = ((x_min + x_max) / 2) / image_width
    y_center = ((y_min + y_max) / 2) / image_height
    width = (x_max - x_min) / image_width
    height = (y_max - y_min) / image_height
    values = [class_id, x_center, y_center, width, height]
    return f"{values[0]} " + " ".join(f"{value:.8f}" for value in values[1:])


def export_split_to_yolo(split_name: str, image_ids, metadata: pd.DataFrame) -> list[dict]:
    image_out_dir = YOLO_DATASET_DIR / "images" / split_name
    label_out_dir = YOLO_DATASET_DIR / "labels" / split_name
    image_out_dir.mkdir(parents=True, exist_ok=True)
    label_out_dir.mkdir(parents=True, exist_ok=True)

    metadata_by_id = metadata.set_index("ImageId")
    records = []
    for image_id in tqdm(list(image_ids), desc=f"Exporting {split_name}"):
        row = metadata_by_id.loc[image_id]
        image = read_grayscale_png(row.image_path)
        image_height, image_width = image.shape[:2]
        boxes = lung_boxes(row)
        if len(boxes) == 0:
            raise ValueError(f"No lung boxes found for {image_id}")

        image_out_path = image_out_dir / row.image_path.name
        label_out_path = label_out_dir / f"{image_id}.txt"
        shutil.copy2(row.image_path, image_out_path)
        label_out_path.write_text("\n".join(xyxy_to_yolo_line(box, image_width, image_height) for box in boxes) + "\n")
        records.append({
            "split": split_name,
            "ImageId": image_id,
            "image_path": image_out_path,
            "label_path": label_out_path,
            "image_width": image_width,
            "image_height": image_height,
            "n_boxes": len(boxes),
        })
    return records


RECREATE_YOLO_DATASET = True
if RECREATE_YOLO_DATASET and YOLO_DATASET_DIR.exists():
    shutil.rmtree(YOLO_DATASET_DIR)

export_records = []
export_records.extend(export_split_to_yolo("train", train_ids, boxes_df))
export_records.extend(export_split_to_yolo("val", valid_ids, boxes_df))
export_records.extend(export_split_to_yolo("test", test_ids, boxes_df))
export_df = pd.DataFrame(export_records)

DATA_YAML_PATH.parent.mkdir(parents=True, exist_ok=True)
data_yaml = {
    "path": str(YOLO_DATASET_DIR.resolve()),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "names": {0: "lung"},
}
DATA_YAML_PATH.write_text(yaml.safe_dump(data_yaml, sort_keys=False))

print(f"YOLO dataset YAML written to: {DATA_YAML_PATH}")
print(f"YOLO dataset root: {YOLO_DATASET_DIR.resolve()}")
export_df.groupby("split").agg(images=("ImageId", "count"), boxes=("n_boxes", "sum"))


In [ ]:
def yolo_label_file_to_xyxy(label_path: Path, image_width: int, image_height: int) -> np.ndarray:
    boxes = []
    for line in label_path.read_text().replace("\\n", "\n").splitlines():
        if not line.strip():
            continue
        class_id, x_center, y_center, width, height = map(float, line.split())
        x_center *= image_width
        y_center *= image_height
        width *= image_width
        height *= image_height
        boxes.append([
            x_center - width / 2,
            y_center - height / 2,
            x_center + width / 2,
            y_center + height / 2,
        ])
    return np.array(boxes, dtype=np.float32)


preview_record = export_df.sample(1, random_state=SEED).iloc[0]
preview_image = read_grayscale_png(preview_record.image_path)
preview_boxes = yolo_label_file_to_xyxy(preview_record.label_path, preview_record.image_width, preview_record.image_height)

fig, ax = plt.subplots(1, 1, figsize=(7, 7))
ax.imshow(normalize_percentile(preview_image), cmap="gray")
draw_boxes(ax, preview_boxes, color="cyan")
ax.set_title(f"YOLO-label round trip: {preview_record.ImageId} ({preview_record.split})")
ax.axis("off")
plt.tight_layout()


## 5. Train the Detection Model

This section defines the training settings and fine-tunes a pretrained detector for a single anatomical class.

A compact model is used to support efficient teaching and rapid iteration. More computationally intensive configurations can be explored after the basic workflow has been established.

In [ ]:
YOLO_MODEL = "yolo26n.pt"
# YOLO_MODEL = "yolo26n.yaml"  # offline architecture-only smoke test, no pretrained weights

EPOCHS = 25
IMAGE_SIZE = 640
BATCH_SIZE = 8
NUM_WORKERS = 2 if IN_COLAB else 4
CONFIDENCE_THRESHOLD = 0.10
EVALUATION_IOU_THRESHOLD = 0.50

DEVICE_ARG = 0 if torch.cuda.is_available() else "cpu"
YOLO_PROJECT_DIR = RESULTS_DIR / "yolo_detection_runs"
YOLO_RUN_NAME = "yolo26n_montgomery_lung_boxes"

print(f"YOLO model: {YOLO_MODEL}")
print(f"Ultralytics dataset config: {DATA_YAML_PATH}")
print(f"Ultralytics device argument: {DEVICE_ARG}")


In [ ]:
model = YOLO(YOLO_MODEL)
train_results = model.train(
    data=str(DATA_YAML_PATH.resolve()),
    epochs=EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    device=DEVICE_ARG,
    project=str(YOLO_PROJECT_DIR.resolve()),
    name=YOLO_RUN_NAME,
    exist_ok=True,
    seed=SEED,
    workers=NUM_WORKERS,
    plots=True,
    cos_lr=True,
    single_cls=True,
)


## 6. Test Evaluation

After training, the framework saves model checkpoints. The checkpoint selected on the validation set is used for final evaluation on the held-out test set.

This section loads the best available checkpoint and evaluates it on the held-out test split from the generated YAML. The test split is used here because it was not used to fit the model.

In [ ]:
RUN_DIR = YOLO_PROJECT_DIR / YOLO_RUN_NAME

candidate_run_dirs = [RUN_DIR]
if "train_results" in globals() and hasattr(train_results, "save_dir"):
    candidate_run_dirs.append(Path(train_results.save_dir))
candidate_run_dirs.extend(sorted(Path.cwd().glob(f"runs/detect/**/{YOLO_RUN_NAME}")))

weights_for_eval = None
for candidate_run_dir in dict.fromkeys(candidate_run_dirs):
    for candidate_weight in [candidate_run_dir / "weights" / "best.pt", candidate_run_dir / "weights" / "last.pt"]:
        if candidate_weight.exists():
            RUN_DIR = candidate_run_dir
            weights_for_eval = candidate_weight
            break
    if weights_for_eval is not None:
        break

searched_dirs = "\n".join(str(path / "weights") for path in candidate_run_dirs)
assert weights_for_eval is not None, f"No trained YOLO weights found. Searched:\n{searched_dirs}"

trained_model = YOLO(str(weights_for_eval))
test_metrics = trained_model.val(
    data=str(DATA_YAML_PATH),
    split="test",
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    device=DEVICE_ARG,
    conf=CONFIDENCE_THRESHOLD,
    iou=EVALUATION_IOU_THRESHOLD,
    plots=True,
)

print(f"Evaluated weights: {weights_for_eval}")
print(f"Evaluation confidence threshold: {CONFIDENCE_THRESHOLD:.2f}; IoU threshold: {EVALUATION_IOU_THRESHOLD:.2f}")
print(test_metrics)


## 7. Preview Predictions

This section provides a visual assessment of model behavior on held-out images. For each selected test image, the reference boxes are compared with the predicted boxes.

The number of predicted boxes is not forced to be two. A well-performing model should usually detect both lungs, but it may return one box if one lung is missed, or more than two boxes if there are duplicate detections. The confidence threshold controls which predictions are displayed. The held-out evaluation reports precision, recall, mAP at IoU 0.50, and mAP averaged across IoU thresholds from 0.50 to 0.95; it uses the confidence and IoU thresholds printed in the preceding cell.

In [ ]:
def result_boxes_xyxy(result) -> tuple[np.ndarray, np.ndarray]:
    if result.boxes is None or len(result.boxes) == 0:
        return np.empty((0, 4), dtype=np.float32), np.empty((0,), dtype=np.float32)
    boxes = result.boxes.xyxy.detach().cpu().numpy().astype(np.float32)
    scores = result.boxes.conf.detach().cpu().numpy().astype(np.float32)
    return boxes, scores


preview_records = export_df.loc[export_df["split"] == "test"].head(4).reset_index(drop=True)
preview_paths = preview_records["image_path"].tolist()
results = trained_model.predict(
    source=[str(path) for path in preview_paths],
    imgsz=IMAGE_SIZE,
    conf=CONFIDENCE_THRESHOLD,
    device=DEVICE_ARG,
    save=False,
    verbose=False,
)

n = len(results)
fig, axes = plt.subplots(n, 2, figsize=(10, 4 * n))
if n == 1:
    axes = axes[None, :]

for i, result in enumerate(results):
    record = preview_records.iloc[i]
    image_path = Path(record.image_path)
    image = read_grayscale_png(image_path)
    true_boxes = yolo_label_file_to_xyxy(record.label_path, record.image_width, record.image_height)
    pred_boxes, pred_scores = result_boxes_xyxy(result)

    axes[i, 0].imshow(normalize_percentile(image), cmap="gray")
    draw_boxes(axes[i, 0], true_boxes, color="lime")
    axes[i, 0].set_title(f"{record.ImageId} ground truth")

    axes[i, 1].imshow(normalize_percentile(image), cmap="gray")
    draw_boxes(axes[i, 1], pred_boxes, color="red", labels=["lung"] * len(pred_boxes), scores=pred_scores)
    axes[i, 1].set_title(f"YOLO predictions ({len(pred_boxes)} boxes)")

    for ax in axes[i]:
        ax.axis("off")
plt.tight_layout()


## 8. Training Curves

Training metrics are recorded during optimization and summarized as performance and loss curves. These curves help assess whether the model improved during training and whether optimization was stable.

The final cell plots the most useful detection metrics and losses when they are available. In an educational setting, these plots are useful for connecting the training process to measurable model performance.

In [ ]:
RESULTS_CSV = RUN_DIR / "results.csv"
assert RESULTS_CSV.exists(), f"Training results CSV not found: {RESULTS_CSV}"

history_df = pd.read_csv(RESULTS_CSV)
history_df.columns = [column.strip() for column in history_df.columns]
display(history_df.tail())

metric_columns = [
    column for column in history_df.columns
    if column in {"metrics/mAP50(B)", "metrics/mAP50-95(B)", "metrics/precision(B)", "metrics/recall(B)"}
]
loss_columns = [column for column in history_df.columns if column.endswith("loss")]

if metric_columns:
    history_df.plot(x="epoch", y=metric_columns, figsize=(8, 4))
    plt.grid(True, alpha=0.3)
    plt.tight_layout()

if loss_columns:
    history_df.plot(x="epoch", y=loss_columns, figsize=(8, 4))
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
